In [0]:
%run ../Streaming_source/NB_generate_streaming_data

In [0]:
from pyspark.sql.functions import current_timestamp, col

In [0]:
def load_to_gold_without_joins(target_catalog_name:str, silver_schema:str, gold_schema:str, columns:list, target_table_name:str, source_name:str):
    """
    Loads the silver table to the gold table

    Args:
        target_catalog_name (str): name of the catalog
        silver_schema (str): name of the silver schema
        gold_schema (str): name of the gold schema
        source_name (str, optional): name of the silver table
        columns (list): list of columns to be used in the gold table
        target_table_name (str): name of the gold table
    """

    main_df = spark.table(f"`{target_catalog_name}`.`{silver_schema}`.`{source_name}_main`")

    df = main_df.select(*columns)

    df.write.mode("overwrite").saveAsTable(f"`{target_catalog_name}`.`{gold_schema}`.`{source_name}_{target_table_name}`")

In [0]:
def load_bronze_streaming_data(source_catalog_name:str, source_name:str, target_catalog_name:str, bronze_schema:str, table_name:str, num_files:int=10, num_rows_per_file:int=1000):
    """
    Loads streaming data for the given table and source name

    Args:
        source_catalog_name (str): The name of the source catalog
        target_catalog_name (str): The name of the target catalog
        bronze_schema (str): The name of the bronze schema
        table_name (str): The name of the table     
        source_name (str): The name of the source for the data
        table_name (str): The name of the table for the data
        num_files (int): The number of files to generate
        num_rows_per_file (int): The number of rows per file
    Returns:
        landing_path (str): The path to the landing directory
        checkpoint_path (str): The path to the checkpoint directory
        schema_path (str): The path to the schema directory
    """

    landing_path, checkpoint_path, schema_path = generate_streaming_data(source_catalog_name=source_catalog_name, target_catalog_name=target_catalog_name, bronze_schema=bronze_schema, source_name=source_name, table_name=table_name, num_files=num_files, num_rows_per_file=num_rows_per_file)

    bronze_df = (
        spark.readStream.format("cloudFiles") \
        .option("cloudFiles.format", "json") \
        .option("cloudFiles.schemaLocation", schema_path) \
        .option("cloudFiles.inferColumnTypes", "true") \
        .load(landing_path) \
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source_file", col("_metadata.file_path"))
        .writeStream \
        .format("delta") \
        .option("checkpointLocation", checkpoint_path) \
        .trigger(availableNow=True) \
        .table(f"{target_catalog_name}.{bronze_schema}.{source_name}_{table_name}")
    )

    bronze_df.awaitTermination()